# Sprint 2 — Wide & Deep, Neighbour+Delta, and a blend

- Protocol: `E.setup` grouped holdout, seeds 42 and 7.
- Training set: 53,947 unique CRM configurations, consensus targets, unweighted.
- Features: the raw 745 `CRM_*` columns.
- Metrics: `src/metrics.py`, as in sections 8–9 of `sprint2_modeling.ipynb`.
- `solution.csv` is not read.

| model | tracker `exp_id` | definition |
|---|---|---|
| Binary Relevance LightGBM | `br-lgbm` | one LightGBM per label, `mcs=20`, `reg_lambda=1`, threshold 0.5 |
| Binary Relevance LightGBM + rules | `br-lgbm-rules` | the above, then `data.apply_additive_rules`; equals the team champion `E5-reg-mcs20-t0.5` |
| Wide & Deep | `wide-deep-512` | neural network — wide linear path plus a shared 512-unit hidden layer, early stopping on EMR |
| Neighbour + Delta | `neighbour-delta` | nearest CRM configuration, then each differing pack's known BIL delta |
| Blend | `blend-wd-nd-lgbm` | equal average of the three probability matrices, threshold 0.5 |

- The 42 additive rules were mined on all of `train.csv`, validation rows included.


## Setup

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from scipy.stats import binomtest

from src import data, experiment as E, metrics as M
from src.models import BinaryRelevanceLGBM
from src.models_nndelta import NeighbourDelta
from src.models_widedeep import WideDeepMultiLabel

# data.load() (called inside E.setup) builds data/cache/arrays.npz from five Sprint 1 CSVs
# on first use, all-or-nothing. Fail with a readable message instead of a FileNotFoundError
# raised deep inside build_cache() after a 700 MB read.
_needed = ['train.csv', 'test.csv', 'train_clean.csv', 'train_clean_v2.csv', 'test_clean_v2.csv']
_missing = [f for f in _needed if not os.path.exists(os.path.join(data.DATA_DIR, f))]
if _missing and not os.path.exists(data.CACHE):
    raise FileNotFoundError(
        f"src/data.py needs {_missing} in data/ to build {data.CACHE}. Regenerate them with "
        "notebooks/sprint1_preprocessing_v3.ipynb, or copy data/cache/arrays.npz from a teammate.")

print('team loader cache :', data.CACHE, '| exists:', os.path.exists(data.CACHE))
print('team tracker      :', E.RESULTS)

team loader cache : C:\Users\andre\ECQ_Sprint2\ECQ\data\cache\arrays.npz | exists: True
team tracker      : C:\Users\andre\ECQ_Sprint2\ECQ\data\cache\results.jsonl


## Configuration

- Constructors take keyword arguments only.
- `E.log` does not enforce unique `exp_id`; records are distinguished by `params['split_seed']`.


In [2]:
SEEDS = (42, 7)
VAL_SIZE = 0.2
THR = 0.5
PIPELINE = 'raw'                # the 745 raw CRM_* columns, i.e. data.features(d, 'raw')
RARE_MAX_TRAIN_ROWS = 200       # team convention: a CRM pack in < 200 training ROWS is rare
TOP_OFFENDERS = 10
MC_SEED = 42                    # seed used for the paired McNemar test
CMP_PATH = '../data/derived/sprint2_andre_comparison.csv'

CHAMP_PARAMS = dict(min_child_samples=20, reg_lambda=1.0)
WD_PARAMS = dict(hidden=(512,), dropout=0.2, wide=True, lr=2e-3,
                 batch_size=256, max_epochs=80, patience=8)
NND_PARAMS = dict(min_pairs=2, min_rate=0.5, max_nb=5)

EXP_CHAMP = 'br-lgbm'
EXP_CHAMP_RULES = 'br-lgbm-rules'
EXP_WD = 'wide-deep-512'
EXP_NND = 'neighbour-delta'
EXP_BLEND = 'blend-wd-nd-lgbm'
ORDER = [EXP_CHAMP, EXP_CHAMP_RULES, EXP_WD, EXP_NND, EXP_BLEND]

# E.log appends to results.jsonl and does not enforce unique exp_ids, so the two seeds
# produce one record per (exp_id, split_seed), told apart by params['split_seed'].
pd.set_option('display.width', 160)

## One seed, end to end

- `E.setup` returns the grouped holdout and the consensus targets.
- `E.dedup_configs` returns unique CRM configurations. Its third value, multiplicity, is unused — weighting by it costs ~21 EMR points.
- Metrics are computed per model at fit time; only the summaries are retained.
- `ok_cfg` records correctness per unique validation configuration. All rows of a configuration receive identical predictions.


In [3]:
def run_seed(split_seed):
    """Fit, score, log and fully profile every candidate for one holdout seed."""
    d = E.setup(seed=split_seed, val_size=VAL_SIZE)
    tr, va = d['tr'], d['va']
    X, Y_cons = d['X_train'], d['Y_cons']
    X_tr, Y_tr, _multiplicity = E.dedup_configs(X[tr], Y_cons[tr])
    X_va, Y_va = X[va], Y_cons[va]

    # one representative row per unique validation configuration
    _, cfg_first = np.unique(d['groups'][va], return_index=True)
    # pack frequency counted in training ROWS, as in the team notebook
    freq = X[tr].sum(0)
    rare = (freq > 0) & (freq < RARE_MAX_TRAIN_ROWS)
    has_rare = X_va[:, rare].any(1)

    print(f"split_seed {split_seed}: train {len(X_tr):,} unique configs (from {len(tr):,} rows) | "
          f"val {len(X_va):,} rows / {len(cfg_first):,} configs | {int(rare.sum())} rare packs")

    bundles = {}

    def record(exp_id, pred, proba, fit_s, family, params, notes):
        res = E.score(d, pred)
        E.log(exp_id, PIPELINE, family, dict(params, split_seed=split_seed), res, fit_s, notes=notes)
        summary, cm, f1 = M.multilabel_report(Y_va, pred, proba)
        err = M.row_errors(Y_va, pred)
        bundles[exp_id] = dict(
            summary=summary, cm=cm, f1=f1, err=err,
            offenders=M.top_offending_columns(Y_va, pred, d['bil_cols'], top_n=TOP_OFFENDERS),
            ok_cfg=(err[cfg_first] == 0), res=res)

    specs = [
        (EXP_CHAMP, 'lgbm-br', lambda: BinaryRelevanceLGBM(CHAMP_PARAMS),
         dict(CHAMP_PARAMS, model_seed=None),
         'BR LightGBM, mcs=20, reg_lambda=1, threshold 0.5, no rules'),
        (EXP_WD, 'widedeep', lambda: WideDeepMultiLabel(seed=split_seed, **WD_PARAMS),
         dict(WD_PARAMS, model_seed=split_seed),
         'wide additive path + shared 512-unit hidden layer'),
        (EXP_NND, 'nndelta', lambda: NeighbourDelta(**NND_PARAMS),
         dict(NND_PARAMS, model_seed=None),
         'nearest CRM config, then apply the per-pack BIL delta'),
    ]

    P = {}
    for exp_id, family, make, params, notes in specs:
        with E.Timer() as timer:
            model = make().fit(X_tr, Y_tr)
            P[exp_id] = np.asarray(model.predict_proba(X_va), dtype=np.float32)
        pred = (P[exp_id] >= THR).astype(np.uint8)
        record(exp_id, pred, P[exp_id], timer.s, family, params, notes)

        if exp_id == EXP_CHAMP:
            # the team's champion definition: overwrite the 42 deterministic BIL columns.
            # Those rules were mined on all of train.csv, so this LEAKS validation labels --
            # reported to quantify the leak, never to choose a model.
            record(EXP_CHAMP_RULES,
                   data.apply_additive_rules(pred, X_va, d['crm_cols'], d['bil_cols']),
                   P[exp_id], timer.s, family, params,
                   'champion + 42 Section 11 additive rules (LEAK: mined on all of train.csv)')

    members = [EXP_WD, EXP_NND, EXP_CHAMP]
    P_blend = np.mean([P[m] for m in members], axis=0)
    record(EXP_BLEND, (P_blend >= THR).astype(np.uint8), P_blend, 0.0, 'blend',
           dict(members=members, weights='equal', thr=THR),
           'equal average of the wd / nndelta / champion probabilities')
    del P, P_blend

    return dict(bundles=bundles, has_rare=has_rare, n_rare_packs=int(rare.sum()),
                rare_per_row_val=float(X_va[:, rare].sum(1).mean()),
                rare_per_row_test=float(d['X_test'][:, rare].sum(1).mean()),
                n_val_rows=len(va), n_val_cfg=len(cfg_first), n_train_cfg=len(X_tr))

## Run both seeds

- ~25–40 min per seed: 731 LightGBM fits, up to 80 Wide & Deep epochs, micro AUCs over ~27M cells per report.
- `E.log` appends after each model completes.


In [4]:
records = {}
for split_seed in SEEDS:
    print(f"\n{'=' * 78}\nsplit_seed {split_seed}\n{'=' * 78}")
    records[split_seed] = run_seed(split_seed)


split_seed 42


split_seed 42: train 53,947 unique configs (from 151,057 rows) | val 36,385 rows / 13,487 configs | 349 rare packs


C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: U

C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: U

...

  ep  30 inner-EMR 0.6917 bce 0.04840 (65s) *


  ep  35 inner-EMR 0.6915 bce 0.05011 (75s)


[wide-deep-512] raw widedeep EMR=86.9803% HL=0.002571 d1=1.577% t=84s wide additive path + shared 512-unit hidden layer


[neighbour-delta] raw nndelta EMR=85.4212% HL=0.002225 d1=2.142% t=31s nearest CRM config, then apply the per-pack BIL delta


[blend-wd-nd-lgbm] raw blend EMR=87.2945% HL=0.001454 d1=1.583% t=0s equal average of the wd / nndelta / champion probabilities


## Per-model report — team metric set

Per model × seed:

- `M.multilabel_report` summary — EMR, Hamming loss, micro and macro F1, precision, recall, ROC-AUC, PR-AUC
- cell confusion matrix; false negatives per false positive
- per-label F1 — median, count = 1, count < 0.8
- wrong bits per row — 0…5, 6+
- `M.top_offending_columns`, top 10
- rare-pack split — rows containing a CRM pack seen in under 200 training rows, against all-frequent rows


In [5]:
for split_seed in SEEDS:
    rec = records[split_seed]
    for exp_id in ORDER:
        b = rec['bundles'][exp_id]
        summary, cm, f1, err = b['summary'], b['cm'], b['f1'], b['err']
        print(f"\n{'=' * 78}\n{exp_id}  |  split_seed {split_seed}\n{'=' * 78}")
        display(summary.to_frame(f'{exp_id} / seed {split_seed}').style.format('{:.5f}'))
        display(cm.style.format('{:,}'))
        print(f"false negatives per false positive: {cm.iloc[0, 1] / max(cm.iloc[1, 0], 1):.1f}")
        print(f"per-label F1 over {len(f1)} columns with positives: median {np.median(f1):.3f} | "
              f"F1 = 1: {(f1 == 1).sum()} | F1 < 0.8: {(f1 < 0.8).sum()}")
        print('wrong bits per validation row:',
              pd.Series(np.minimum(err, 6)).value_counts().sort_index()
                .rename(index={6: '6+'}).to_dict())
        print(f'\ncolumns that alone break an otherwise perfect row (top {TOP_OFFENDERS}):')
        display(b['offenders'].round(4))

        hr = rec['has_rare']
        display(pd.DataFrame({
            'val rows': [hr.sum(), (~hr).sum()],
            'EMR': [(err[hr] == 0).mean(), (err[~hr] == 0).mean()],
            'share with 6+ wrong bits': [(err[hr] >= 6).mean(), (err[~hr] >= 6).mean()],
        }, index=[f'contains a CRM pack with <{RARE_MAX_TRAIN_ROWS} train rows',
                  'all packs frequent']).round(4))

    print(f"\nseed {split_seed}: {rec['n_rare_packs']} rare CRM packs | mean per row: "
          f"validation {rec['rare_per_row_val']:.3f}, test {rec['rare_per_row_test']:.3f}")


br-lgbm  |  split_seed 42


,br-lgbm / seed 42
EMR,0.87654
bit accuracy,0.99867
Hamming loss,0.00133
F1 micro,0.96697
F1 samples,0.97367
F1 macro,0.37482
median per-label F1,0.22581
precision micro,0.99394
recall micro,0.94142
ROC-AUC micro,0.98572


,predicted 1,predicted 0
actual 1,"516,562","32,142"
actual 0,"3,151","26,045,580"


false negatives per false positive: 10.2
per-label F1 over 731 columns with positives: median 0.226 | F1 = 1: 0 | F1 < 0.8: 547
wrong bits per validation row: {0: 31893, 1: 757, 2: 313, 3: 220, 4: 238, 5: 253, '6+': 2711}

columns that alone break an otherwise perfect row (top 10):


,bil_column,n_rows_sole_error,missed_positive,false_positive,col_error_rate_all_rows,share_of_dist1_rows
0,BIL_SUBSCRIBER_STATUS_ORIG_Active,47,47,0,0.0050,0.0621
1,BIL_3940_PACK,44,36,8,0.0028,0.0581
2,BIL_4289_PACK,36,36,0,0.0021,0.0476
3,BIL_4297_PACK,31,31,0,0.0021,0.0410
4,BIL_3803_PACK,22,22,0,0.0022,0.0291
5,BIL_3812_PACK,16,16,0,0.0016,0.0211
6,BIL_3855_PACK,11,11,0,0.0016,0.0145
7,BIL_3952_PACK,11,9,2,0.0021,0.0145
8,BIL_4156_PACK,10,10,0,0.0016,0.0132
9,BIL_3999_PACK,10,10,0,0.0019,0.0132


...

,predicted 1,predicted 0
actual 1,"475,623","31,022"
actual 0,"4,147","23,682,384"


false negatives per false positive: 7.5
per-label F1 over 731 columns with positives: median 0.262 | F1 = 1: 0 | F1 < 0.8: 541
wrong bits per validation row: {0: 28891, 1: 524, 2: 242, 3: 227, 4: 226, 5: 273, '6+': 2713}

columns that alone break an otherwise perfect row (top 10):


,bil_column,n_rows_sole_error,missed_positive,false_positive,col_error_rate_all_rows,share_of_dist1_rows
0,BIL_3940_PACK,60,54,6,0.0036,0.1145
1,BIL_3803_PACK,25,25,0,0.0024,0.0477
2,BIL_4136_PACK,18,0,18,0.0015,0.0344
3,BIL_3762_PACK,11,1,10,0.0027,0.0210
4,BIL_3801_PACK,10,10,0,0.0015,0.0191
5,BIL_4118_PACK,10,10,0,0.0013,0.0191
6,BIL_4102_PACK,9,9,0,0.0022,0.0172
7,BIL_SUBSCRIBER_STATUS_ORIG_Active,9,9,0,0.0036,0.0172
8,BIL_4115_PACK,9,9,0,0.0015,0.0172
9,BIL_4311_PACK,8,8,0,0.0017,0.0153


,val rows,EMR,share with 6+ wrong bits
contains a CRM pack with <200 train rows,4157,0.1034,0.6372
all packs frequent,28939,0.9835,0.0022



seed 7: 355 rare CRM packs | mean per row: validation 0.480, test 0.039


## Comparison table

- Rows = models, columns = (seed, metric).
- Written to `data/derived/sprint2_andre_comparison.csv`.


In [6]:
def summary_row(b):
    s, cm, err = b['summary'], b['cm'], b['err']
    return {
        'EMR': s['EMR'],
        'Hamming loss': s['Hamming loss'],
        'F1 micro': s['F1 micro'],
        'F1 macro': s['F1 macro'],
        'precision micro': s['precision micro'],
        'recall micro': s['recall micro'],
        'FN per FP': cm.iloc[0, 1] / max(cm.iloc[1, 0], 1),
        'median label F1': s['median per-label F1'],
        'share 6+ wrong bits': float((err >= 6).mean()),
    }


comparison = pd.concat(
    {f'seed {s}': pd.DataFrame({e: summary_row(records[s]['bundles'][e]) for e in ORDER}).T
     for s in SEEDS}, axis=1).reindex(ORDER)
comparison.index.name = 'exp_id'

os.makedirs(os.path.dirname(CMP_PATH), exist_ok=True)
comparison.to_csv(CMP_PATH)
print('written:', CMP_PATH, comparison.shape)
display(comparison.style.format('{:.5f}'))

written:

 ../data/derived/sprint2_andre_comparison.csv (5, 18)


## Paired exact McNemar vs the champion, seed 42

- Unit: the 13,487 unique CRM configurations.
- `b` and `c` are the discordant configurations; exact binomial test against p = 0.5.
- Reference: Binary Relevance LightGBM without rules.
- `EMR configs` is unweighted per configuration; the EMR above is row-weighted.


In [7]:
ref = records[MC_SEED]['bundles'][EXP_CHAMP]['ok_cfg']

rows = []
for exp_id in ORDER:
    if exp_id == EXP_CHAMP:
        continue
    a = records[MC_SEED]['bundles'][exp_id]['ok_cfg']
    only_a, only_ref = int((a & ~ref).sum()), int((~a & ref).sum())
    n_disc = only_a + only_ref
    rows.append({
        'exp_id': exp_id,
        'configs only this right (b)': only_a,
        'configs only champion right (c)': only_ref,
        'discordant': n_disc,
        'EMR configs': a.mean(),
        'dEMR configs vs champion': a.mean() - ref.mean(),
        'McNemar p': binomtest(only_a, n_disc, 0.5).pvalue if n_disc else 1.0,
    })

mcnemar = pd.DataFrame(rows).set_index('exp_id')
print(f"reference {EXP_CHAMP} | {len(ref):,} unique validation configurations | "
      f"champion EMR(configs) {ref.mean():.4f}")
display(mcnemar.style.format({'EMR configs': '{:.4f}',
                              'dEMR configs vs champion': '{:+.4f}',
                              'McNemar p': '{:.3g}'}))

reference br-lgbm | 13,487 unique validation configurations | champion EMR(configs) 0.6824


,configs only this right (b),configs only champion right (c),discordant,EMR configs,dEMR configs vs champion,McNemar p
exp_id,,,,,,
br-lgbm-rules,74,7,81,0.6873,+0.0050,3.17e-15
wide-deep-512,256,150,406,0.6902,+0.0079,1.61e-07
neighbour-delta,301,535,836,0.6650,-0.0174,5.01e-16
blend-wd-nd-lgbm,247,56,303,0.6965,+0.0142,8.96e-30


## Results

Validation EMR (%):

| model | seed 42 | seed 7 |
|---|---|---|
| Binary Relevance LightGBM | 87.65 | 86.30 |
| Binary Relevance LightGBM + rules *(= team champion)* | 87.84 | 86.49 |
| Wide & Deep | 87.93 | 86.98 |
| Neighbour + Delta | 81.50 | 85.40 |
| Blend | **88.26** | **87.29** |

Paired tests against the team champion — `data/derived/sprint2_andre_significance.csv`:

| model | seed 42 ΔEMR pts (95% CI) | seed 7 ΔEMR pts (95% CI) | CI excludes zero |
|---|---|---|---|
| Blend | +0.42 (+0.13, +0.72) | +0.80 (+0.47, +1.23) | both seeds |
| Blend + rules | +0.45 (+0.16, +0.74) | +0.82 (+0.49, +1.26) | both seeds |
| Wide & Deep | +0.09 (−0.18, +0.31) | +0.49 (+0.15, +0.93) | seed 7 only |
| Neighbour + Delta | −6.35 (−14.41, −1.25) | −1.07 (−1.86, −0.43) | both seeds, negative |

## Observations

- The champion is reproduced to six decimals: 0.878411 on seed 42, 0.864938 on seed 7.
- The blend exceeds the champion on both seeds; McNemar and the bootstrap CI agree.
- Wide & Deep on seed 42: McNemar p = 0.045, CI includes zero. 200 configurations won, 161 lost; the lost ones cover more rows.
- The 42 additive rules add +0.19 EMR to the champion and +0.022 to the blend.
- Neighbour + Delta is the lowest single model and is uniquely correct on 301 configurations the champion misses.
- 69.5% of BIL labels have fewer than 100 positives in the validation fold. Macro F1 ≈ 0.38, micro F1 ≈ 0.97.
- Wide & Deep: FN/FP 1.2, median label F1 0.151, EMR 87.93. Champion: FN/FP 10.2, median label F1 0.226, EMR 87.65.
- Rows containing a pack seen in under 200 training rows: EMR ≈ 6%, and ≈ 67% carry 6+ wrong bits. All-frequent rows: EMR ≈ 98%.
- Wide & Deep training: BCE rose 0.025 → 0.051 while inner-holdout EMR rose 0.50 → 0.69.

## Caveats

- Two splits.
- Neighbour + Delta predictions differ by 7–21 rows between environments; cause not established.
- Runs were logged as `pipeline='raw'` and relabelled to `raw+consensus` after execution.
